In [14]:
import tensorflow as tf

In [15]:
IMG_SIZE = 224
BATCH_SIZE = 32

In [16]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train_data",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224,224),
    batch_size=16
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train_data",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=16
)

Found 118 files belonging to 2 classes.


Using 95 files for training.
Found 118 files belonging to 2 classes.
Using 23 files for validation.


In [17]:
normalization_layer=tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))





In [18]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    # 🔥 Added deeper feature learning
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),

    layers.BatchNormalization(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    # 🔥 Final output
    layers.Dense(2, activation='softmax')
])

In [19]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
history=model.fit(train_ds,
                  validation_data=val_ds,
                  epochs=50)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 292ms/step - accuracy: 0.6211 - loss: 0.7947 - val_accuracy: 0.8696 - val_loss: 0.5677
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 294ms/step - accuracy: 0.6526 - loss: 0.7296 - val_accuracy: 0.8696 - val_loss: 0.5625
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 278ms/step - accuracy: 0.6526 - loss: 0.7512 - val_accuracy: 0.8696 - val_loss: 0.5575
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step - accuracy: 0.5789 - loss: 0.8348 - val_accuracy: 0.8696 - val_loss: 0.5524
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 279ms/step - accuracy: 0.6105 - loss: 0.7281 - val_accuracy: 0.8696 - val_loss: 0.5583
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 274ms/step - accuracy: 0.6211 - loss: 0.7176 - val_accuracy: 0.8696 - val_loss: 0.5578
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step - accuracy: 0.6316 - loss: 0.7243 - val_accuracy: 0.8696 - val_loss: 0.5566
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 275ms/step - accuracy: 0.6105 - loss: 0.7439 - val_accuracy: 0.8696 - val_loss:

In [22]:
model.save("model.h5")
model.save("model.keras")

In [23]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,170,539 (19.72 MB)

 Trainable params: 372,546 (1.42 MB)

 Non-trainable params: 4,052,899 (15.46 MB)

 Optimizer params: 745,094 (2.84 MB)

In [24]:
import os

for cls in ['defective', 'non-defective']:
    path = f"data"   # adjust path
    print(cls, len(os.listdir(path)))

defective 2
non-defective 2
